# Imports

In [1]:
import os
import sys
sys.path.append("..")

In [2]:
import re

In [3]:
from AGoTI.model import ApiVLLMModel
import AGoTI.operations as op
import AGoTI.GoO as goo

# Model

In [4]:
API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
model_name = "yagpt5lite"

In [5]:
model = ApiVLLMModel(API_KEY, API_URL, model_name)

# Graph

In [6]:
initial_prompt = [
    {
        "role": "user",
        "content": "Suggest a theme and a couple of criteria for a very short scientific paper.\n"
            "Write your response in format below. Follow it strictly!\n"
            "Theme: write a theme here\n"
            "Criteria:\n"
            "1. Criteria number 1\n"
            "... \n"
            "n. Criteria number n\n"
    }
]

In [7]:
class InitialGenerator(op.SimplePromptGenerator):
    def parse_generation(self, text):
        matches = re.findall(
            r"(Theme:\s.+\n+Criteria:\s*\n(?:.|\n)*)", text)
        if matches is None:
            return []
        return [matches[0]]

In [8]:
class PlanGenerator(op.SimpleForwardGenerator):
    async def thought_collector(self):
        for parent in self.parents:
            thoughts = op.one_thought_waiter(self.subscribtions, parent)
            async for thought in thoughts:
               yield  ([thought], {"text": thought.text})

    def make_prompt(self, text):
        return [
            {"role": "user", "content": "Based on the description below "
            "write a short plan for a scientific paper:\n"
            f"{text}"}
            ]

In [9]:
root = InitialGenerator(
    model,
    initial_prompt,
    name="InitialGenerator",
    description="Generates theme and description for a scientific paper"
    )
generate_plan = PlanGenerator(
    model,
    parents=[root],
    name="PlanGenerator",
    description="Generates plan of scientific paper based on the theme and description provided"
    )

In [10]:
graph = goo.GraphOfOperations(
    [root, generate_plan],
    [root],
    [generate_plan]
)

# Inference

In [11]:
result = await graph.run()
print(result[0])

 **Plan for a scientific paper on the theme: «The impact of artificial intelligence on the job market»**

**I. Introduction**

1.1. Brief overview of the current state of artificial intelligence (AI) and its penetration into various sectors of the economy.
1.2. Statement of the research problem: how does AI affect the job market, including job creation, job displacement, and the skills required for the workforce?
1.3. Objectives of the research: to analyze the impact of AI on the job market, to identify trends, and to propose potential solutions to mitigate negative consequences.

**II. Literature review**

2.1. Overview of existing research on the impact of AI on the job market.
2.2. Identification of key themes and arguments in the literature.
2.3. Critique of the existing research, highlighting gaps and limitations.

**III. Methodology**

3.1. Description of the research methods used, including data sources (e.g., government reports, industry studies, academic papers).
3.2. Explanat